# Kalshi Orderbook Ingestion — 15-Minute UP/DOWN Markets
**Real-time Orderbook Polling → DuckDB**

Polls the orderbook REST API for **BTC, ETH, SOL, XRP 15-minute UP/DOWN markets** and stores depth-of-book snapshots in DuckDB.

### Why Orderbook vs WebSocket?
- **Complete liquidity picture**: Full bid/ask depth at every price level
- **No asks needed**: In binary markets, yes bid at 7¢ = no ask at 93¢
- **Easier to work with**: Simple REST polling vs complex WebSocket state management

### Market Format
- **Series**: KXBTC15M, KXETH15M, KXSOL15M, KXXRP15M
- **Markets**: 
  - `KXBTC15M-26FEB251000-00` = BTC UP (expires 10:00)
  - `KXBTC15M-26FEB251000-01` = BTC DOWN (expires 10:00)

### Setup
```bash
pip install requests duckdb cryptography python-dotenv
```

### Credentials
Create a `.env` file:
```
KALSHI_KEY_ID=your_api_key_id
KALSHI_PRIVATE_KEY=private_key.pem
KALSHI_ENV=prod
```

### Important Notes
- **Network Access**: This notebook requires network access to call Kalshi's API
- **Fallback Mode**: If API discovery fails, you can manually specify active markets in the discovery cell
- **Update Manual Tickers**: Check https://kalshi.com/markets/kxbtc15m for current active 15-min markets



---
## 1. Imports & Config

In [1]:
import asyncio
import base64
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import duckdb
import requests
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import padding
from dotenv import load_dotenv

load_dotenv()

# ── Kalshi API endpoints ──────────────────────────────────────────────────────
API_URLS = {
    "prod": "https://api.elections.kalshi.com/trade-api/v2",
    "demo": "https://demo-api.kalshi.co/trade-api/v2",
}

KALSHI_ENV       = os.getenv("KALSHI_ENV", "prod")
KEY_ID           = os.getenv("KALSHI_KEY_ID", "")
PRIVATE_KEY_PATH = os.getenv("KALSHI_PRIVATE_KEY", "private_key.pem")
DB_PATH          = os.getenv("KALSHI_DB_PATH", "kalshi_orderbooks.duckdb")
API_BASE         = API_URLS[KALSHI_ENV]

# ── 15-Minute UP/DOWN Markets ─────────────────────────────────────────────────
TRACKED_SERIES = ["KXBTC15M", "KXETH15M", "KXSOL15M", "KXXRP15M"]

# ── Polling config ────────────────────────────────────────────────────────────
POLL_INTERVAL_SEC = 1      # Poll orderbook every 2 seconds
ORDERBOOK_DEPTH   = 10     # Get top 10 levels (0 = all levels)

# ── Manual Ticker Fallback ────────────────────────────────────────────────────
# Set to empty list [] to disable fallback
# If API discovery fails, manually specify active markets here
MANUAL_TICKERS = []  # Will be set in Market Discovery cell

print(f"Environment : {KALSHI_ENV}")
print(f"API Base    : {API_BASE}")
print(f"Database    : {DB_PATH}")
print(f"Poll Rate   : every {POLL_INTERVAL_SEC}s")
print(f"Key ID set  : {'YES' if KEY_ID else 'NO'}")
print(f"PEM exists  : {'YES' if Path(PRIVATE_KEY_PATH).exists() else 'NO'}")


Environment : prod
API Base    : https://api.elections.kalshi.com/trade-api/v2
Database    : kalshi_orderbooks.duckdb
Poll Rate   : every 1s
Key ID set  : YES
PEM exists  : YES


---
## 2. Auth Helpers

In [2]:
def load_private_key(path: str):
    """Load RSA private key from PEM file."""
    with open(path, "rb") as f:
        return serialization.load_pem_private_key(f.read(), password=None)


def sign_request(private_key, method: str, path: str, timestamp: str) -> str:
    """RSA-PSS sign a REST request."""
    message = timestamp + method + path
    sig = private_key.sign(
        message.encode("utf-8"),
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.DIGEST_LENGTH,
        ),
        hashes.SHA256(),
    )
    return base64.b64encode(sig).decode("utf-8")


def make_headers(private_key, method: str, path_with_params: str) -> dict:
    """
    Kalshi V2 Signer:
    1. Sign path WITHOUT query parameters.
    2. Include /trade-api/v2 prefix in the signature string.
    """
    path_only = path_with_params.split('?')[0]
    signing_path = f"/trade-api/v2{path_only}" if not path_only.startswith("/trade-api/v2") else path_only
    
    ts = str(int(time.time() * 1000))
    # Signature: TIMESTAMP + METHOD + PATH
    message = ts + method + signing_path
    
    sig = private_key.sign(
        message.encode("utf-8"),
        padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.DIGEST_LENGTH),
        hashes.SHA256()
    )
    
    return {
        "KALSHI-ACCESS-KEY": KEY_ID,
        "KALSHI-ACCESS-SIGNATURE": base64.b64encode(sig).decode("utf-8"),
        "KALSHI-ACCESS-TIMESTAMP": ts,
        "Content-Type": "application/json"
    }

print("Auth helpers ready.")

Auth helpers ready.


---
## 3. Database Setup

In [3]:
def init_db(db_path: str) -> duckdb.DuckDBPyConnection:
    """Create DuckDB with orderbook snapshot table."""
    con = duckdb.connect(db_path)

    # Orderbook snapshots with full depth
    con.execute("""
        CREATE TABLE IF NOT EXISTS orderbook_snapshots (
            id                  BIGINT PRIMARY KEY,
            fetched_at          TIMESTAMPTZ NOT NULL,
            market_ticker       VARCHAR     NOT NULL,
            asset               VARCHAR     NOT NULL,
            direction           VARCHAR     NOT NULL,
            
            -- Best bid/ask (top of book)
            yes_best_bid_dollars   DOUBLE,
            yes_best_bid_qty       DOUBLE,
            no_best_bid_dollars    DOUBLE,
            no_best_bid_qty        DOUBLE,
            
            -- Derived mid price
            mid_dollars         DOUBLE,
            spread_dollars      DOUBLE,
            
            -- Full book depth (JSON arrays)
            yes_bids_json       JSON,      -- [[price, qty], ...]
            no_bids_json        JSON       -- [[price, qty], ...]
        )
    """)

    # Index for time-series queries
    con.execute("""
        CREATE INDEX IF NOT EXISTS idx_orderbook_asset_time
        ON orderbook_snapshots (asset, fetched_at)
    """)

    print(f"Database ready: {db_path}")
    return con


con = init_db(DB_PATH)

Database ready: kalshi_orderbooks.duckdb


In [4]:
import requests
import time

# 2. Update discovery to be more resilient
def get_active_markets(private_key, series_ticker: str) -> list[dict]:
    """Fetch all open markets for a series; handles API state mismatches."""
    path = f"/markets?series_ticker={series_ticker}&status=open&limit=100"
    headers = make_headers(private_key, "GET", path)
    
    try:
        resp = requests.get(f"{API_BASE}{path}", headers=headers, timeout=10)
        resp.raise_for_status()
        # The API returns 'active' in JSON but 'open' was the query param.
        return resp.json().get("markets", [])
    except Exception as e:
        print(f"[ERROR] Discovery failed for {series_ticker}: {e}")
        return []

def parse_ticker(ticker: str) -> tuple[str, str]:
    """Extract asset and suffix/strike from ticker."""
    for series in TRACKED_SERIES:
        if ticker.startswith(series):
            asset = series.replace("KX", "").replace("15M", "")
            # Suffix is the part after the last dash (e.g., '00', '01', or '45')
            suffix = ticker.split('-')[-1]
            direction = f"STRIKE-{suffix}" if suffix not in ["00", "01"] else ("UP" if suffix == "00" else "DOWN")
            return asset, direction
    return "UNKNOWN", "NONE"

def discover_all_markets(private_key) -> list[str]:
    """Finds ALL live tickers for tracked crypto series."""
    tickers = []
    # REMOVED: print(f"[DISCOVERY] Scanning {TRACKED_SERIES}...")
    
    for series in TRACKED_SERIES:
        markets = get_active_markets(private_key, series)
        for m in markets:
            ticker = m.get("ticker")
            if ticker:
                tickers.append(ticker)
    
    if not tickers and MANUAL_TICKERS:
        return MANUAL_TICKERS
    return tickers

print("Market discovery functions ready.")

Market discovery functions ready.


---
## 5. Orderbook Fetcher

In [5]:
def fetch_orderbook(private_key, ticker: str, depth: int = 10) -> dict:
    """Fetch orderbook using the v2 'orderbook' key (not orderbook_fp)."""
    path = f"/markets/{ticker}/orderbook" # No depth param in standard v2 orderbook call
    headers = make_headers(private_key, "GET", path)
    
    resp = requests.get(f"{API_BASE}{path}", headers=headers)
    resp.raise_for_status()
    # V2 returns {'orderbook': {'yes': [[p, q]...], 'no': [[p, q]...]}}
    return resp.json().get("orderbook", {})

def parse_orderbook_snapshot(ticker: str, orderbook: dict) -> dict:
    """Flatten book into DuckDB row format."""
    asset, direction = parse_ticker(ticker)
    yes_bids = orderbook.get("yes", [])
    no_bids = orderbook.get("no", [])
    
    # Best prices (highest bid for each side)
    y_best = max(yes_bids, key=lambda x: x[0]) if yes_bids else [None, None]
    n_best = max(no_bids, key=lambda x: x[0]) if no_bids else [None, None]
    
    mid = (float(y_best[0]) + (100 - float(n_best[0]))) / 2 if (y_best[0] and n_best[0]) else None
    
    return {
        "market_ticker": ticker, "asset": asset, "direction": direction,
        "yes_best_bid_dollars": float(y_best[0])/100 if y_best[0] else None,
        "yes_best_bid_qty": float(y_best[1]) if y_best[1] else None,
        "no_best_bid_dollars": float(n_best[0])/100 if n_best[0] else None,
        "no_best_bid_qty": float(n_best[1]) if n_best[1] else None,
        "mid_dollars": mid/100 if mid else None,
        "spread_dollars": (100 - float(y_best[0]) - float(n_best[0]))/100 if (y_best[0] and n_best[0]) else None,
        "yes_bids_json": json.dumps(yes_bids),
        "no_bids_json": json.dumps(no_bids),
    }
    
print("Orderbook fetcher ready.")

Orderbook fetcher ready.


---
## 6. Polling Loop

In [6]:
async def poll_orderbooks(con, private_key, stop_event):
    """
    Main polling loop:
    1. Discover active markets every 60s
    2. Poll each market's orderbook every POLL_INTERVAL_SEC
    3. Store snapshots in DuckDB
    """
    active_tickers = []
    last_discovery = 0
    snapshot_id = int(time.time() * 1e6)  # Unique ID generator
    
    print("[START] Orderbook polling started")
    
    while not stop_event.is_set():
        try:
            # Rediscover markets every 60 seconds
            now = time.time()
            if now - last_discovery > 5:
                active_tickers = discover_all_markets(private_key)
                last_discovery = now
                print(f"[DISCOVERY] Found {len(active_tickers)} active markets")
            
            if not active_tickers:
                print("[WARN] No active markets found, waiting...")
                await asyncio.sleep(10)
                continue
            
            # Poll each market
            fetched_at = datetime.now(timezone.utc)
            
            for ticker in active_tickers:
                try:
                    orderbook = fetch_orderbook(private_key, ticker, ORDERBOOK_DEPTH)
                    snapshot = parse_orderbook_snapshot(ticker, orderbook)
                    
                    # Insert into DB
                    snapshot_id += 1
                    con.execute("""
                        INSERT INTO orderbook_snapshots VALUES (
                            ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?
                        )
                    """, [
                        snapshot_id,
                        fetched_at,
                        snapshot["market_ticker"],
                        snapshot["asset"],
                        snapshot["direction"],
                        snapshot["yes_best_bid_dollars"],
                        snapshot["yes_best_bid_qty"],
                        snapshot["no_best_bid_dollars"],
                        snapshot["no_best_bid_qty"],
                        snapshot["mid_dollars"],
                        snapshot["spread_dollars"],
                        snapshot["yes_bids_json"],
                        snapshot["no_bids_json"],
                    ])
                    
                    asset = snapshot["asset"]
                    direction = snapshot["direction"]
                    mid = snapshot["mid_dollars"]
                    spread = snapshot["spread_dollars"]
                    print(f"[SNAP] {asset} {direction:>4} | mid=${mid:.4f} spread=${spread:.4f}" if mid else f"[SNAP] {asset} {direction:>4} | no quotes")
                    
                except Exception as e:
                    print(f"[ERROR] Failed to fetch {ticker}: {e}")
            
            # Wait before next poll
            await asyncio.sleep(POLL_INTERVAL_SEC)
            
        except Exception as e:
            print(f"[ERROR] Polling error: {e}")
            await asyncio.sleep(5)
    
    print("[STOP] Orderbook polling stopped")


print("Polling loop ready.")

Polling loop ready.


---
## 7. Start Polling

In [7]:
# Validate credentials
assert KEY_ID, "Set KALSHI_KEY_ID in .env"
assert Path(PRIVATE_KEY_PATH).exists(), f"Private key not found: {PRIVATE_KEY_PATH}"

private_key = load_private_key(PRIVATE_KEY_PATH)
stop_event = asyncio.Event()

# Run as background task
poll_task = asyncio.ensure_future(
    poll_orderbooks(con, private_key, stop_event)
)

print("Orderbook polling started. Run STOP cell to shut down.")

Orderbook polling started. Run STOP cell to shut down.


[START] Orderbook polling started
[DISCOVERY] Found 4 active markets
[SNAP] BTC STRIKE-30 | mid=$0.2800 spread=$0.0400
[SNAP] ETH STRIKE-30 | mid=$0.3400 spread=$0.0400
[SNAP] SOL STRIKE-30 | mid=$0.2850 spread=$0.0300
[SNAP] XRP STRIKE-30 | mid=$0.4350 spread=$0.0500
[SNAP] BTC STRIKE-30 | mid=$0.2700 spread=$0.0400
[SNAP] ETH STRIKE-30 | mid=$0.3300 spread=$0.0400
[SNAP] SOL STRIKE-30 | mid=$0.2950 spread=$0.0500
[SNAP] XRP STRIKE-30 | mid=$0.4300 spread=$0.0400
[DISCOVERY] Found 4 active markets
[SNAP] BTC STRIKE-30 | mid=$0.2700 spread=$0.0400
[SNAP] ETH STRIKE-30 | mid=$0.3350 spread=$0.0500
[SNAP] SOL STRIKE-30 | mid=$0.2950 spread=$0.0500
[SNAP] XRP STRIKE-30 | mid=$0.4250 spread=$0.0300
[SNAP] BTC STRIKE-30 | mid=$0.2550 spread=$0.0100
[SNAP] ETH STRIKE-30 | mid=$0.3000 spread=$0.0400
[SNAP] SOL STRIKE-30 | mid=$0.2600 spread=$0.0400
[SNAP] XRP STRIKE-30 | mid=$0.3500 spread=$0.0400
[DISCOVERY] Found 4 active markets
[SNAP] BTC STRIKE-30 | mid=$0.2750 spread=$0.0300
[SNAP] ETH 

In [12]:
import shutil

snapshot_dir = 'db_snapshot'
if os.path.exists(snapshot_dir):
    shutil.rmtree(snapshot_dir)

con.execute(f"EXPORT DATABASE '{snapshot_dir}' (FORMAT PARQUET)")
print(f"Snapshot exported to {snapshot_dir}/")

Snapshot exported to db_snapshot/


---
## 8. Stop Polling

In [13]:
stop_event.set()
await asyncio.sleep(1)
print("Polling stopped.")

Polling stopped.


In [ ]:
# # Delete if needed
# con.close()
# import os
# if os.path.exists(DB_PATH):
#     os.remove(DB_PATH)
#     print(f"Deleted {DB_PATH}")